In [ ]:
# Federated Credit Card Fraud Detection with Flower
# Setup: install dependencies (run once per environment)
import sys, subprocess, pkgutil

required = [
    "flwr==1.9.0",
    "torch",
    "scikit-learn",
    "pandas",
    "numpy",
    "matplotlib",
    "seaborn",
    "tqdm"
]

missing = [p for p in required if pkgutil.find_loader(p.split("==")[0]) is None]
if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
else:
    print("All dependencies already installed.")

print("Python:", sys.version)



In [ ]:
# Data download, preprocessing, and federated partitioning
import os
import urllib.request
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

DATA_URL = "https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv"
DATA_PATH = os.path.join(os.getcwd(), "creditcard.csv")

if not os.path.exists(DATA_PATH):
    print("Downloading dataset...")
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)
    print("Downloaded to", DATA_PATH)
else:
    print("Dataset already present at", DATA_PATH)

# Load data
raw = pd.read_csv(DATA_PATH)
print(raw.shape, raw["Class"].value_counts(normalize=True))

# Features and labels
X = raw.drop(columns=["Class"]).values.astype(np.float32)
y = raw["Class"].values.astype(np.int64)

# Standardize features
scaler = StandardScaler()
X = scaler.fit_transform(X).astype(np.float32)

# Hold out a global test set for evaluation of global model
X_train_full, X_test_global, y_train_full, y_test_global = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Compute class weights (for severe imbalance)
classes = np.unique(y_train_full)
class_weights = compute_class_weight(
    class_weight="balanced", classes=classes, y=y_train_full
)
class_weight_tensor = {int(c): float(w) for c, w in zip(classes, class_weights)}
print("Class weights:", class_weight_tensor)

# Create federated client partitions
NUM_CLIENTS = 5  # adjust for demo
np.random.seed(42)

# Stratified split into clients: we maintain class ratio per client
client_indices = [[] for _ in range(NUM_CLIENTS)]
for cls in classes:
    cls_idx = np.where(y_train_full == cls)[0]
    np.random.shuffle(cls_idx)
    splits = np.array_split(cls_idx, NUM_CLIENTS)
    for i in range(NUM_CLIENTS):
        client_indices[i].extend(splits[i].tolist())

# Shuffle per-client indices
for i in range(NUM_CLIENTS):
    rng = np.random.default_rng(seed=100 + i)
    rng.shuffle(client_indices[i])

# Build per-client datasets
clients_data = []
for i in range(NUM_CLIENTS):
    ci = np.array(client_indices[i])
    Xc, yc = X_train_full[ci], y_train_full[ci]
    X_train_c, X_val_c, y_train_c, y_val_c = train_test_split(
        Xc, yc, test_size=0.2, random_state=42, stratify=yc
    )
    clients_data.append({
        "train": (X_train_c, y_train_c),
        "val": (X_val_c, y_val_c),
    })

print("Clients built:", [len(c["train"][1]) for c in clients_data], "val:", [len(c["val"][1]) for c in clients_data])

# Persist artifacts needed later
artifact = {
    "X_test_global": X_test_global,
    "y_test_global": y_test_global,
    "clients_data": clients_data,
    "class_weight_tensor": class_weight_tensor,
    "input_dim": X.shape[1],
}



In [ ]:
# PyTorch model and training utilities
import math
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
INPUT_DIM = artifact["input_dim"]
CLASS_WEIGHTS = artifact["class_weight_tensor"]

class MLP(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(32, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(1)


def make_loader(X: np.ndarray, y: np.ndarray, batch_size: int, shuffle: bool) -> DataLoader:
    X_t = torch.from_numpy(X)
    y_t = torch.from_numpy(y.astype(np.float32))
    ds = TensorDataset(X_t, y_t)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


def train_one_epoch(model: nn.Module, loader: DataLoader, optimizer: torch.optim.Optimizer) -> float:
    model.train()
    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([CLASS_WEIGHTS[1]], dtype=torch.float32, device=DEVICE)
    )
    epoch_loss = 0.0
    for xb, yb in loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * xb.size(0)
    return epoch_loss / len(loader.dataset)


def evaluate(model: nn.Module, loader: DataLoader):
    model.eval()
    all_logits, all_y = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE)
            logits = model(xb)
            all_logits.append(logits.cpu())
            all_y.append(yb)
    logits = torch.cat(all_logits).numpy()
    y_true = torch.cat(all_y).numpy()
    y_prob = 1.0 / (1.0 + np.exp(-logits))
    try:
        roc = roc_auc_score(y_true, y_prob)
    except ValueError:
        roc = float("nan")
    try:
        apr = average_precision_score(y_true, y_prob)
    except ValueError:
        apr = float("nan")
    # threshold 0.5 for accuracy
    y_pred = (y_prob >= 0.5).astype(np.int64)
    acc = (y_pred == y_true).mean()
    return {"roc_auc": float(roc), "avg_precision": float(apr), "accuracy": float(acc)}


# Small sanity check loader for global test later
GLOBAL_TEST_LOADER = make_loader(
    artifact["X_test_global"], artifact["y_test_global"], batch_size=1024, shuffle=False
)

print("Using device:", DEVICE)



In [ ]:
# Flower client and federated simulation
import flwr as fl
from typing import Dict, Tuple, Optional, OrderedDict

# Keep the initial global model to be shared across clients
GLOBAL_MODEL = MLP(INPUT_DIM).to(DEVICE)

def get_parameters(model: nn.Module):
    return [val.cpu().numpy() for _, val in model.state_dict().items()]


def set_parameters(model: nn.Module, parameters) -> None:
    state_dict = model.state_dict()
    new_state_dict = OrderedDict()
    for (k, _), v in zip(state_dict.items(), parameters):
        new_state_dict[k] = torch.tensor(v)
    model.load_state_dict(new_state_dict, strict=True)


class FraudClient(fl.client.NumPyClient):
    def __init__(self, cid: int, train_set: Tuple[np.ndarray, np.ndarray], val_set: Tuple[np.ndarray, np.ndarray]):
        self.cid = cid
        self.model = MLP(INPUT_DIM).to(DEVICE)
        self.train_loader = make_loader(train_set[0], train_set[1], batch_size=1024, shuffle=True)
        self.val_loader = make_loader(val_set[0], val_set[1], batch_size=2048, shuffle=False)
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=1e-3)

    def get_parameters(self, config):
        return get_parameters(self.model)

    def fit(self, parameters, config):
        set_parameters(self.model, parameters)
        epochs = int(config.get("local_epochs", 1))
        for _ in range(epochs):
            train_one_epoch(self.model, self.train_loader, self.optimizer)
        metrics = evaluate(self.model, self.val_loader)
        return get_parameters(self.model), len(self.train_loader.dataset), metrics

    def evaluate(self, parameters, config):
        set_parameters(self.model, parameters)
        metrics = evaluate(self.model, self.val_loader)
        # Flower expects (loss, num_examples, metrics). We provide dummy loss as 1 - roc.
        loss = 1.0 - (metrics.get("roc_auc") or 0.0)
        return float(loss), len(self.val_loader.dataset), metrics


def client_fn(cid: str):
    idx = int(cid)
    data = artifact["clients_data"][idx]
    return FraudClient(idx, data["train"], data["val"]).to_client()


def get_evaluate_fn():
    # Use separate model instance for global evaluation on the fixed global test set
    def evaluate_fn(server_round: int, parameters, config):
        model = MLP(INPUT_DIM).to(DEVICE)
        set_parameters(model, parameters)
        metrics = evaluate(model, GLOBAL_TEST_LOADER)
        loss = 1.0 - (metrics.get("roc_auc") or 0.0)
        print(f"Round {server_round} — Global ROC-AUC={metrics['roc_auc']:.4f}, APR={metrics['avg_precision']:.4f}, ACC={metrics['accuracy']:.4f}")
        return float(loss), metrics
    return evaluate_fn

# Strategy
strategy = fl.server.strategy.FedAvg(
    fraction_fit=1.0,
    fraction_evaluate=1.0,
    min_fit_clients=len(artifact["clients_data"]),
    min_evaluate_clients=len(artifact["clients_data"]),
    min_available_clients=len(artifact["clients_data"]),
    evaluate_fn=get_evaluate_fn(),
    on_fit_config_fn=lambda rnd: {"local_epochs": 1 if rnd < 3 else 2},
)

# Run in-process simulation (single machine)
history = fl.simulation.start_simulation(
    client_fn=client_fn,
    num_clients=len(artifact["clients_data"]),
    config=fl.server.ServerConfig(num_rounds=5),
    strategy=strategy,
)

# Save final global parameters
final_params = history.global_model_parameters
print("Simulation complete.")



In [ ]:
# Centralized baseline training and evaluation
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, precision_recall_curve, confusion_matrix

# Build loaders on the centralized training set (merge all clients' train parts)
X_train_merged = np.concatenate([c["train"][0] for c in artifact["clients_data"]], axis=0)
y_train_merged = np.concatenate([c["train"][1] for c in artifact["clients_data"]], axis=0)

train_loader_central = make_loader(X_train_merged, y_train_merged, batch_size=2048, shuffle=True)

def train_centralized(epochs: int = 5):
    model = MLP(INPUT_DIM).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    for ep in range(epochs):
        loss = train_one_epoch(model, train_loader_central, optimizer)
        m = evaluate(model, GLOBAL_TEST_LOADER)
        print(f"Centralized epoch {ep+1}: loss={loss:.4f}, ROC-AUC={m['roc_auc']:.4f}, APR={m['avg_precision']:.4f}, ACC={m['accuracy']:.4f}")
    return model

central_model = train_centralized(epochs=5)
central_metrics = evaluate(central_model, GLOBAL_TEST_LOADER)
print("Centralized final:", central_metrics)

# Curves
central_model.eval()
probs = []
truth = []
with torch.no_grad():
    for xb, yb in GLOBAL_TEST_LOADER:
        xb = xb.to(DEVICE)
        logits = central_model(xb)
        p = 1.0 / (1.0 + torch.exp(-logits))
        probs.append(p.cpu().numpy())
        truth.append(yb.numpy())
probs = np.concatenate(probs)
truth = np.concatenate(truth)

fpr, tpr, _ = roc_curve(truth, probs)
prec, rec, _ = precision_recall_curve(truth, probs)

plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(fpr, tpr, label=f"ROC-AUC={central_metrics['roc_auc']:.4f}")
plt.plot([0,1],[0,1],'k--')
plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title("Centralized ROC Curve"); plt.legend()

plt.subplot(1,2,2)
plt.plot(rec, prec, label=f"AP={central_metrics['avg_precision']:.4f}")
plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title("Centralized PR Curve"); plt.legend()
plt.show()

# Confusion matrix at 0.5
cm = confusion_matrix(truth, (probs>=0.5).astype(int))
plt.figure(figsize=(4,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Centralized Confusion Matrix (thr=0.5)')
plt.xlabel('Predicted'); plt.ylabel('True')
plt.show()



In [ ]:
# Federated learning metrics over rounds (if available)
# Flower History may contain centralized metrics. We try to plot ROC-AUC across rounds.
try:
    hist = history
    # In recent Flower versions, metrics_centralized is a dict: {"evaluate": [(rnd, metric_dict), ...]}
    eval_points = hist.metrics_centralized.get("evaluate", []) if hasattr(hist, "metrics_centralized") else []
    rounds = []
    rocs = []
    for r, m in eval_points:
        rounds.append(r)
        rocs.append(m.get("roc_auc"))
    if rounds:
        plt.figure(figsize=(6,4))
        plt.plot(rounds, rocs, marker='o')
        plt.xlabel("Round")
        plt.ylabel("Global ROC-AUC")
        plt.title("Federated ROC-AUC across rounds")
        plt.grid(True)
        plt.show()
    else:
        print("No centralized evaluation history available. See printed per-round metrics above.")
except Exception as e:
    print("Could not extract history:", e)



### How to run this demo

- Run each cell from top to bottom.
- The dataset (`creditcard.csv`) will download automatically.
- The FL simulation runs locally with 5 clients and 5 rounds. Per-round global ROC-AUC is printed.
- Then the centralized baseline trains and plots ROC/PR and a confusion matrix.

### What to demonstrate live

- **Federated setup**: Explain clients keep their data local; only model weights are shared. Point at the `FraudClient` implementation.
- **Severe class imbalance**: Show `Class` distribution and how class weights handle it.
- **Per-round improvement**: Highlight printed ROC-AUC across FL rounds and the line plot.
- **Benchmark**: Compare FL performance trend vs centralized baseline curves.
- **Resource story**: All clients simulated on one machine here; in production each bank/device would run a client.

### Talking points

- **Privacy**: No raw transactions leave the client; only model updates do.
- **Robust metrics**: Use ROC-AUC and Average Precision (PR-AUC) as fraud is highly imbalanced.
- **Aggregation**: FedAvg with configurable local epochs; discuss trade-off between communication and convergence.
- **Limitations**: Non-IID data can slow convergence; we used stratified splits for demo.

### Extensions (nice-to-have)

- Add Differential Privacy (per-client noise via `opacus`).
- Add Secure Aggregation / TLS.
- Introduce non-IID partitions (by time or merchant category) and compare.
- Swap model to Gradient Boosted Trees (via federated approaches) or a deeper NN.

